**EDA**

In [2]:
import random
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords, wordnet as wn

nltk.download("stopwords")
nltk.download("omw-1.4")

STOPWORDS = set(stopwords.words("indonesian"))

# ======================
# NORMALISASI
# ======================
def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ======================
# WORDNET
# ======================
def get_synonyms(word):
    synonyms = set()
    for syn in wn.synsets(word, lang="ind"):
        for lemma in syn.lemmas(lang="ind"):
            s = lemma.name().replace("_", " ").lower()
            if s != word:
                synonyms.add(s)
    return list(synonyms)

# ======================
# EDA FUNCTIONS
# ======================
def synonym_replacement(words, alpha):
    new_words = words.copy()
    candidates = [w for w in words if w not in STOPWORDS]
    random.shuffle(candidates)
    n = max(1, int(alpha * len(words)))

    replaced = 0
    for word in candidates:
        syns = get_synonyms(word)
        if syns:
            new_words = [random.choice(syns) if w == word else w for w in new_words]
            replaced += 1
        if replaced >= n:
            break
    return new_words

def random_insertion(words, alpha):
    new_words = words.copy()
    n = max(1, int(alpha * len(words)))
    for _ in range(n):
        candidates = [w for w in new_words if w not in STOPWORDS]
        if not candidates:
            break
        word = random.choice(candidates)
        syns = get_synonyms(word)
        if syns:
            new_words.insert(random.randint(0, len(new_words)), random.choice(syns))
    return new_words

def random_swap(words, alpha):
    new_words = words.copy()
    n = max(1, int(alpha * len(words)))
    for _ in range(n):
        if len(new_words) >= 2:
            i, j = random.sample(range(len(new_words)), 2)
            new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words

def random_deletion(words, alpha):
    if len(words) == 1:
        return words
    new_words = [w for w in words if random.random() > alpha]
    return new_words if new_words else [random.choice(words)]

def eda(sentence, alpha=0.1):
    words = sentence.split()
    return [
        " ".join(synonym_replacement(words, alpha)),
        " ".join(random_insertion(words, alpha)),
        " ".join(random_swap(words, alpha)),
        " ".join(random_deletion(words, alpha)),
    ]

# ======================
# MAIN PIPELINE
# ======================
df = pd.read_csv(r"C:\SKRIPSI BRYAN\Tahap 2\2A\data_70_2A.csv")
df["Text"] = df["Text"].apply(normalize)

df_0 = df[df["label"] == 0]
df_1 = df[df["label"] == 1]

N0 = len(df_0)

targets = {
    "60": int(0.6 * N0),
    "80": int(0.8 * N0),
    "100": N0
}

augmented = df_1.copy()
saved = set()

for _, row in df_1.iterrows():
    if len(augmented) >= max(targets.values()):
        break

    aug_texts = eda(row["Text"], alpha=0.1)
    for txt in aug_texts:
        augmented = pd.concat([
            augmented,
            pd.DataFrame([[1, txt]], columns=["label", "Text"])
        ], ignore_index=True)

        for k, target in targets.items():
            if len(augmented) >= target and k not in saved:
                final_df = pd.concat([df_0, augmented.iloc[:target]], ignore_index=True)
                final_df.to_csv(f"EDA_70_{k}.csv", index=False)
                print(f"✅ Saved EDA_70_{k}.csv")
                saved.add(k)

        if len(saved) == len(targets):
            break

# ======================
# PRINT CONTOH AUGMENTASI
# ======================

print("\n" + "="*60)
print("CONTOH HASIL AUGMENTASI (10 DATA)")
print("="*60)

sample_df = df_1.sample(n=min(10, len(df_1)), random_state=42)

for idx, row in sample_df.iterrows():
    original = row["Text"]
    augmented_versions = eda(original, alpha=0.1)

    print("\n[ASLI]")
    print(original)

    print("[SR]")
    print(augmented_versions[0])

    print("[RI]")
    print(augmented_versions[1])

    print("[RS]")
    print(augmented_versions[2])

    print("[RD]")
    print(augmented_versions[3])

    print("-"*60)

[nltk_data] Downloading package stopwords to C:\Users\Evzen
[nltk_data]     Ozora\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Evzen
[nltk_data]     Ozora\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


✅ Saved EDA_70_60.csv
✅ Saved EDA_70_80.csv
✅ Saved EDA_70_100.csv

CONTOH HASIL AUGMENTASI (10 DATA)

[ASLI]
aku merasa sudah tidak ada lagi yang tersisa untukku selamat tinggal
[SR]
aku merasa sudah tidak ada lagi yang tersisa untukku selamat lambat
[RI]
aku merasa sudah tidak ada lagi yang tersisa untukku selamat tinggal
[RS]
aku merasa tersisa tidak ada lagi yang sudah untukku selamat tinggal
[RD]
merasa sudah tidak lagi yang untukku selamat tinggal
------------------------------------------------------------

[ASLI]
aku dilahirkan dengan cacat mengapa aku harus dilahirkan aku tidak ingin dilahirkan di dunia ini
[SR]
aku dilahirkan dengan pengembara mengapa aku harus dilahirkan aku tidak ingin dilahirkan di dunia ini
[RI]
aku dilahirkan dengan cacat mengapa aku harus dilahirkan aku tidak ingin dilahirkan di pencemaran dunia ini
[RS]
aku dilahirkan aku cacat mengapa aku harus dilahirkan dengan tidak ingin dilahirkan di dunia ini
[RD]
aku dilahirkan dengan cacat mengapa aku harus dil

**SMOTE**

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE

# ======================
# NORMALISASI
# ======================
def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"C:\SKRIPSI BRYAN\Tahap 2\2A\data_70_2A.csv")
df["Text"] = df["Text"].apply(normalize)

df_0 = df[df["label"] == 0]
df_1 = df[df["label"] == 1]

N0 = len(df_0)

targets = {
    "60": int(0.6 * N0),
    "80": int(0.8 * N0),
    "100": N0
}

# ======================
# TF-IDF
# ======================
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3)
)

X = vectorizer.fit_transform(df["Text"])
y = df["label"]

feature_names = np.array(vectorizer.get_feature_names_out())

# ======================
# SMOTE FUNCTION
# ======================
def smote_and_save(target_ratio, tag):
    target_n = targets[tag]

    sampling_strategy = {
        0: N0,
        1: target_n
    }

    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=42,
        k_neighbors=min(5, len(df_1)-1)
    )

    X_res, y_res = smote.fit_resample(X, y)

    # ambil hanya hasil label 1
    mask = (y_res.to_numpy() == 1)
    X_label1 = X_res[mask]

    X_syn = X_label1[len(df_1):]


    # inverse TF-IDF → teks
    texts = []
    for row in X_syn:
        indices = row.toarray().flatten().argsort()[-10:][::-1]
        words = feature_names[indices]
        texts.append(" ".join(words))

    df_syn = pd.DataFrame({
        "label": 1,
        "Text": texts
    })

    final_df = pd.concat([
        df_0,
        df_1,
        df_syn
    ], ignore_index=True)

    final_df.to_csv(f"smote_70_{tag}.csv", index=False)
    print(f"✅ Saved smote_70_{tag}.csv")

    return df_syn

# ======================
# RUN UNTUK 60 / 80 / 100
# ======================
synthetic_examples = {}

for tag in ["60", "80", "100"]:
    synthetic_examples[tag] = smote_and_save(targets[tag], tag)

# ======================
# PRINT CONTOH HASIL
# ======================
print("\n" + "="*60)
print("CONTOH HASIL SMOTE + TF-IDF (10 DATA)")
print("="*60)

sample_original = df_1.sample(n=min(10, len(df_1)), random_state=42)
sample_smote = synthetic_examples["100"].sample(
    n=min(10, len(synthetic_examples["100"])),
    random_state=42
)

for i in range(len(sample_original)):
    print("\n[ASLI]")
    print(sample_original.iloc[i]["Text"])

    print("[SMOTE + TFIDF]")
    print(sample_smote.iloc[i]["Text"])

    print("-"*60)

✅ Saved smote_70_60.csv
✅ Saved smote_70_80.csv
✅ Saved smote_70_100.csv

CONTOH HASIL SMOTE + TF-IDF (10 DATA)

[ASLI]
aku merasa sudah tidak ada lagi yang tersisa untukku selamat tinggal
[SMOTE + TFIDF]
alasan punya terus nggak buat hidup udah aku nyali terasa
------------------------------------------------------------

[ASLI]
aku dilahirkan dengan cacat mengapa aku harus dilahirkan aku tidak ingin dilahirkan di dunia ini
[SMOTE + TFIDF]
ak du ancrit aja mati bs gk mah gini kek
------------------------------------------------------------

[ASLI]
sudah bertahan lama menahan semuanya sendiri saatnya untuk menyerah dan berakhir disini
[SMOTE + TFIDF]
aku ingin kekacauan maafkan hanya dalam dan mati tidak tahun
------------------------------------------------------------

[ASLI]
cape banget sama dunia kalau mau mati aja emang gabisa ya
[SMOTE + TFIDF]
tahan aku semua ini hanya berakhir tidak ingin akan lagi
------------------------------------------------------------

[ASLI]
lisavirjola

**BACKTRANSLATION**

In [7]:
import pandas as pd
import random
import torch
import os
import json
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import NllbTokenizer

# ================================
# DEVICE
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ================================
# LOAD NLLB MODEL
# ================================
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

# Languages for backtranslation
LANGS = [
    "eng_Latn", "spa_Latn", "fra_Latn", "zho_Hans", "deu_Latn",
    "ita_Latn", "por_Latn", "jpn_Jpan", "rus_Cyrl", "kor_Hang",
    "arb_Arab", "hin_Deva"
]

src_lang = "ind_Latn"
MAX_LEN = 128
BATCH = 16
MAX_RETRY = 12

SAVE_INTERVAL = 200
AUTOSAVE = "bt_autosave.json"


# ================================
# TRANSLATE BATCH
# ================================
def translate_batch(texts, src, tgt):
    tokenizer.src_lang = src
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
            max_length=MAX_LEN
        )

    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]


# ================================
# BACKTRANSLATE WITH RETRY
# ================================
def back_translate_with_retry(texts):
    seen_langs = set()

    for _ in range(MAX_RETRY):
        candidates = [l for l in LANGS if l not in seen_langs]
        if not candidates:
            candidates = LANGS
            seen_langs = set()

        tgt = random.choice(candidates)
        seen_langs.add(tgt)

        fwd = translate_batch(texts, src_lang, tgt)
        back = translate_batch(fwd, tgt, src_lang)

        yield back

    yield None


# ================================
# DUPLICATE CHECK
# ================================
def is_exact_duplicate(ori, aug, seen):
    return (
        not aug or
        aug.strip() == ori.strip() or
        aug in seen
    )


# ================================
# AUTOSAVE
# ================================
def load_autosave():
    if os.path.exists(AUTOSAVE):
        with open(AUTOSAVE, "r") as f:
            data = json.load(f)
        return data.get("augmented", []), data.get("count", 0)

    return [], 0


def save_autosave(aug_list):
    with open(AUTOSAVE, "w") as f:
        json.dump({
            "augmented": aug_list,
            "count": len(aug_list)
        }, f)


# ================================
# SAVE CSV
# ================================
def save_csv(df0, df1_augmented, name):
    df_out = pd.concat([df0, df1_augmented], ignore_index=True)
    df_out.to_csv(name, index=False)
    print(f"[SAVE] {name} created.")


# ================================
# MAIN AUGMENTATION LOOP
# ================================
def augment_label_1(df):

    df0 = df[df.label == 0]
    df1 = df[df.label == 1]

    originals = df1.Text.tolist()
    base_count = len(df0)

    # Target milestones
    target_60 = int(base_count * 0.6)
    target_80 = int(base_count * 0.8)
    target_100 = base_count

    milestones = {
        target_60: "bt_70_60.csv",
        target_80: "bt_70_80.csv",
        target_100: "bt_70_100.csv"
    }

    # Autoload progress
    augmented, _ = load_autosave()
    seen = set(originals + augmented)

    # Count label 1
    count = len(originals) + len(augmented)

    print(f"Loaded autosave: {len(augmented)} augmented items, total label 1 now = {count}")

    idx = 0

    pbar = tqdm(
        total=target_100,
        initial=count,
        unit="aug",
        desc="Backtranslation"
    )

    saved_milestones = set()

    while count < target_100:

        batch = originals[idx: idx + BATCH]

        if not batch:
            idx = 0
            continue

        chosen = None

        for bt in back_translate_with_retry(batch):
            if bt is None:
                break

            if any(not is_exact_duplicate(o, a, seen) for o, a in zip(batch, bt)):
                chosen = bt
                break

        if chosen is None:
            idx += BATCH
            continue

        for ori, aug in zip(batch, chosen):

            if count >= target_100:
                break

            if not is_exact_duplicate(ori, aug, seen):
                augmented.append(aug)
                seen.add(aug)
                count += 1
                pbar.update(1)

            if count % SAVE_INTERVAL == 0:
                save_autosave(augmented)

            # Milestone check
            for target, fname in milestones.items():
                if count >= target and target not in saved_milestones:

                    num_aug_needed = target - len(originals)
                    df1_aug = pd.DataFrame({
                        "label": 1,
                        "Text": originals + augmented[:num_aug_needed]
                    })

                    save_csv(df0, df1_aug, fname)
                    saved_milestones.add(target)

        idx += BATCH

    # Final save
    num_aug_needed = target_100 - len(originals)
    df1_aug = pd.DataFrame({
        "label": 1,
        "Text": originals + augmented[:num_aug_needed]
    })

    save_csv(df0, df1_aug, "bt_70_100.csv")
    save_autosave(augmented)

    pbar.close()

    # ================================
    # PRINT SAMPLE AFTER COMPLETION
    # ================================
    print("\n=== 20 SAMPLE ORIGINAL → RANDOM LANG PER SAMPLE → BACK TO INDO ===\n")

    sample_originals = originals[:20]

    for ori in sample_originals:
        lang = random.choice(LANGS)

        print(f"\n-----------------------------")
        print(f"[Language Used]: {lang}")

        # Forward translation
        fwd = translate_batch([ori], src_lang, lang)[0]

        # Back translation
        back = translate_batch([fwd], lang, src_lang)[0]

        print(f"Original : {ori}")
        print(f"Translated ({lang}) : {fwd}")
        print(f"Back to Indo : {back}")

    print("\n-----------------------------------\n")

    print("All targets completed!")


# ================================
# RUN
# ================================
df = pd.read_csv(r"C:\SKRIPSI BRYAN\Tahap 2\2A\data_70_2A.csv")
df.columns = ["label", "Text"]

augment_label_1(df)

DEVICE: cuda


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 992.71it/s, Materializing param=model.shared.weight]                                    
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded autosave: 0 augmented items, total label 1 now = 2310


Backtranslation:  60%|█████▉    | 2765/4620 [02:03<08:04,  3.83aug/s]  

[SAVE] bt_70_60.csv created.


Backtranslation:  80%|███████▉  | 3689/4620 [08:38<04:27,  3.48aug/s]

[SAVE] bt_70_80.csv created.


Backtranslation: 100%|██████████| 4620/4620 [14:04<00:00,  2.74aug/s]


[SAVE] bt_70_100.csv created.
[SAVE] bt_70_100.csv created.

=== 20 SAMPLE ORIGINAL → RANDOM LANG PER SAMPLE → BACK TO INDO ===


-----------------------------
[Language Used]: eng_Latn
Original : Aku minta maaf kalau aku terlihat putus asa, tapi aku bener-bener pengen menyerah.
Translated (eng_Latn) : I'm sorry if I look desperate, but I really want to give up.
Back to Indo : Maaf jika aku terlihat putus asa, tapi aku benar-benar ingin menyerah.

-----------------------------
[Language Used]: por_Latn
Original : ini antara sy bkl bunuh diri atau realistis
Translated (por_Latn) : Isto é entre suicídio e realista.
Back to Indo : Ini antara bunuh diri dan realistis.

-----------------------------
[Language Used]: kor_Hang
Original : capek bgt anjink mati aja kli y
Translated (kor_Hang) : 죽기엔 너무 피곤해
Back to Indo : Aku terlalu lelah untuk mati.

-----------------------------
[Language Used]: eng_Latn
Original : Mau jual ginjal, apa bunuh diri karena terjerat pinjol,, solusi pinjol apa #BLA

**BERT MASKING**

In [1]:
import pandas as pd
import random
import torch
import os
import json
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

# ================================
# DEVICE
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ================================
# LOAD INDO BERTWEET MLM
# ================================
model_name = "indolem/indobertweet-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
model.eval()

MAX_LEN = 500
BATCH = 16
MAX_RETRY = 5
SAVE_INTERVAL = 200
AUTOSAVE = "mlm_autosave.json"
MASK_PROB = 0.1

SPECIAL_TOKENS = ["[CLS]", "[SEP]", "[PAD]", "[UNK]"]


# ================================
# TEXT NORMALIZATION
# ================================
def normalize_text(s):
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([.,!?;:])", r"\1", s)
    s = s.strip()
    return s


# ================================
# CLEAN WORDPIECE
# ================================
def clean_wordpiece(tokens):
    final = []
    for tok in tokens:
        if tok.startswith("##"):
            if final:
                final[-1] += tok[2:]
            else:
                final.append(tok[2:])
        elif tok in SPECIAL_TOKENS:
            continue
        else:
            final.append(tok)
    return " ".join(final).strip()


# ================================
# MASK TOKENS
# ================================
def mask_tokens(texts, mask_prob=MASK_PROB):
    masked_texts = []
    for text in texts:
        tokens = tokenizer.tokenize(text)
        if not tokens:
            masked_texts.append(text)
            continue

        new_tokens = []
        for t in tokens:

            # Skip @username
            if "@" in t:
                new_tokens.append(t)
                continue

            if random.random() < mask_prob:
                new_tokens.append(tokenizer.mask_token)
            else:
                new_tokens.append(t)

        masked_texts.append(tokenizer.convert_tokens_to_string(new_tokens))
    return masked_texts


# ================================
# MLM AUGMENTATION
# ================================
def mlm_augment(texts):
    for _ in range(MAX_RETRY):
        masked_texts = mask_tokens(texts)

        inputs = tokenizer(
            masked_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits

        augmented_texts = []
        for i, masked in enumerate(masked_texts):
            input_ids = inputs["input_ids"][i]
            logit_i = logits[i]
            new_tokens = []

            for j, tok_id in enumerate(input_ids):
                tok_str = tokenizer.convert_ids_to_tokens(int(tok_id))

                # Replace MASK tokens
                if tok_str == tokenizer.mask_token:
                    pred_id = torch.argmax(logit_i[j]).item()
                    tok_str = tokenizer.convert_ids_to_tokens(pred_id)

                new_tokens.append(tok_str)

            augmented_texts.append(clean_wordpiece(new_tokens))

        yield masked_texts, augmented_texts

    yield None, None


# ================================
# DUPLICATE CHECK
# ================================
def is_exact_duplicate(ori, aug, seen):
    if not aug:
        return True

    n_ori = normalize_text(ori)
    n_aug = normalize_text(aug)

    if n_ori == n_aug:
        return True

    seen_norm = {normalize_text(x) for x in seen}
    return n_aug in seen_norm


# ================================
# AUTOSAVE ORIGINAL–MASKED–AUGMENTED
# ================================
def load_autosave():
    if os.path.exists(AUTOSAVE):
        with open(AUTOSAVE, "r") as f:
            data = json.load(f)
            return data.get("pairs", []), data.get("count", 0)
    return [], 0


def save_autosave(pairs):
    with open(AUTOSAVE, "w") as f:
        json.dump({"pairs": pairs, "count": len(pairs)}, f)


# ================================
# SAVE CSV
# ================================
def save_csv(df0, df1_augmented, name):
    df_out = pd.concat([df0, df1_augmented], ignore_index=True)
    df_out.to_csv(name, index=False)
    print(f"[SAVE] {name} created.")


# ================================
# MAIN LOOP
# ================================
def augment_label_1(df):

    df0 = df[df.label == 0]
    df1 = df[df.label == 1]

    originals = df1.Text.tolist()
    base_count = len(df0)

    target_60 = int(base_count * 0.6)
    target_80 = int(base_count * 0.8)
    target_100 = base_count

    milestones = {
        target_60: "mlm_70_60.csv",
        target_80: "mlm_70_80.csv",
        target_100: "mlm_70_100.csv"
    }

    pairs, _ = load_autosave()
    seen = set(originals + [x["augmented"] for x in pairs])
    count = len(originals) + len(pairs)

    print(f"Loaded autosave: {len(pairs)} augmented pairs, total label 1 = {count}")

    idx = 0
    pbar = tqdm(total=target_100, initial=count, unit="aug", desc="MLM Augmentation")
    saved_milestones = set()

    while count < target_100:

        batch = originals[idx: idx + BATCH]
        if not batch:
            idx = 0
            continue

        chosen = None
        chosen_mask = None

        for masked_batch, aug_texts in mlm_augment(batch):
            if aug_texts is None:
                break
            if any(not is_exact_duplicate(o, a, seen) for o, a in zip(batch, aug_texts)):
                chosen = aug_texts
                chosen_mask = masked_batch
                break

        if chosen is None:
            idx += BATCH
            continue

        # Pair them
        for ori, masked, aug in zip(batch, chosen_mask, chosen):

            if count >= target_100:
                break

            if not is_exact_duplicate(ori, aug, seen):

                pairs.append({
                    "original": ori,
                    "masked": masked,
                    "augmented": aug
                })

                seen.add(aug)
                count += 1
                pbar.update(1)

                if count % SAVE_INTERVAL == 0:
                    save_autosave(pairs)

                # milestone saving
                for target, fname in milestones.items():
                    if count >= target and target not in saved_milestones:
                        need = target - len(originals)
                        aug_texts = [x["augmented"] for x in pairs[:need]]
                        df1_aug = pd.DataFrame({"label": 1, "Text": originals + aug_texts})
                        save_csv(df0, df1_aug, fname)
                        saved_milestones.add(target)

        idx += BATCH

    # Final save
    need = target_100 - len(originals)
    aug_texts = [x["augmented"] for x in pairs[:need]]
    df1_aug = pd.DataFrame({"label": 1, "Text": originals + aug_texts})
    save_csv(df0, df1_aug, "mlm_70_100.csv")
    save_autosave(pairs)
    pbar.close()

    print("All augmentation targets completed!")

    # Final print section
    print("\n===== Contoh 10 ORIGINAL vs MASKED vs AUGMENTED =====\n")
    for pair in pairs[:10]:
        print(f"[ORIGINAL ] {pair['original']}")
        print(f"[MASKED   ] {pair['masked']}")
        print(f"[AUGMENTED] {pair['augmented']}\n")


# ================================
# RUN
# ================================
df = pd.read_csv(r"C:\SKRIPSI BRYAN\Tahap 2\2A\data_70_2A.csv")
df.columns = ["label", "Text"]
augment_label_1(df)

c:\SKRIPSI BRYAN\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cuda


Loading weights: 100%|██████████| 204/204 [00:00<00:00, 1022.23it/s, Materializing param=cls.predictions.transform.dense.weight]                 
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: indolem/indobertweet-base-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when 

Loaded autosave: 0 augmented pairs, total label 1 = 2310


MLM Augmentation:  60%|██████    | 2780/4620 [00:16<01:00, 30.29aug/s]

[SAVE] mlm_70_60.csv created.


MLM Augmentation:  80%|████████  | 3701/4620 [00:55<00:41, 21.94aug/s]

[SAVE] mlm_70_80.csv created.


MLM Augmentation: 100%|██████████| 4620/4620 [01:47<00:00, 21.40aug/s]

[SAVE] mlm_70_100.csv created.
[SAVE] mlm_70_100.csv created.
All augmentation targets completed!

===== Contoh 10 ORIGINAL vs MASKED vs AUGMENTED =====

[ORIGINAL ] Aku minta maaf kalau aku terlihat putus asa, tapi aku bener-bener pengen menyerah.
[MASKED   ] [UNK] minta maaf kalau aku [MASK] putus asa, tapi aku bener - bener [MASK] menyerah.
[AUGMENTED] minta maaf kalau aku pernah putus asa , tapi aku bener - bener pantang menyerah .

[ORIGINAL ] ini antara sy bkl bunuh diri atau realistis
[MASKED   ] ini [MASK] sy [MASK] bunuh diri atau realistis
[AUGMENTED] ini menurut sy mau bunuh diri atau realistis

[ORIGINAL ] capek bgt anjink mati aja kli y
[MASKED   ] capek bgt [MASK] mati aja [MASK] y
[AUGMENTED] capek bgt pgn mati aja kali y

[ORIGINAL ] Mau jual ginjal, apa bunuh diri karena terjerat pinjol,, solusi pinjol apa #BLACKPINK_WORLDTOUR #51stBudgetSpeech #stockmarketcrash #LiverpoolRealMadrid
[MASKED   ] [UNK] jual ginjal, apa bunuh diri karena terjerat pinjol, [MASK] solusi pin

**PARAPHRASE GENERATION**

In [2]:
import pandas as pd
import random
import torch
import os
import json
import re
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ================================
# DEVICE
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# ================================
# LOAD INDO T5 PARAPHRASE
# ================================
model_name = "Wikidepia/IndoT5-base-paraphrase"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

MAX_LEN = 500
BATCH = 8
SAVE_INTERVAL = 200
MAX_RETRY = 5
AUTOSAVE = "t5_paraphrase_autosave.json"

# ================================
# NORMALIZATION
# ================================
def normalize_text(s):
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([.,!?;:])", r"\1", s)
    return s.strip()

# ================================
# DUPLICATE CHECK
# ================================
def is_exact_duplicate(ori, aug, seen):
    if not aug:
        return True

    n_ori = normalize_text(ori)
    n_aug = normalize_text(aug)

    if n_ori == n_aug:
        return True

    seen_norm = {normalize_text(x) for x in seen}
    return n_aug in seen_norm

# ================================
# AUTOSAVE
# ================================
def load_autosave():
    if os.path.exists(AUTOSAVE):
        with open(AUTOSAVE, "r") as f:
            data = json.load(f)
            return data.get("pairs", []), data.get("count", 0)
    return [], 0

def save_autosave(pairs):
    with open(AUTOSAVE, "w") as f:
        json.dump({"pairs": pairs, "count": len(pairs)}, f)

# ================================
# SAVE CSV
# ================================
def save_csv(df0, df1_augmented, name):
    df_out = pd.concat([df0, df1_augmented], ignore_index=True)
    df_out.to_csv(name, index=False)
    print(f"[SAVE] {name} created.")

# ================================
# PARAPHRASE GENERATION
# ================================
def generate_paraphrases(texts, num_return_sequences=1, num_beams=4):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LEN,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            num_return_sequences=1,
            repetition_penalty=1.5
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return decoded

# ================================
# MAIN LOOP
# ================================
def augment_label_1(df):

    df0 = df[df.label == 0]
    df1 = df[df.label == 1]

    originals = df1.Text.tolist()
    base_count = len(df0)

    target_60 = int(base_count * 0.6)
    target_80 = int(base_count * 0.8)
    target_100 = base_count

    milestones = {
        target_60: "t5_70_60.csv",
        target_80: "t5_70_80.csv",
        target_100: "t5_70_100.csv"
    }

    pairs, _ = load_autosave()
    seen = set(originals + [x["augmented"] for x in pairs])
    count = len(originals) + len(pairs)

    print(f"Loaded autosave: {len(pairs)} augmented pairs, total label 1 = {count}")

    idx = 0
    pbar = tqdm(total=target_100, initial=count, unit="aug", desc="T5 Paraphrase")
    saved_milestones = set()

    while count < target_100:

        batch = originals[idx: idx + BATCH]
        if not batch:
            idx = 0
            continue

        chosen_aug = None

        # attempt generate paraphrase up to MAX_RETRY
        for _ in range(MAX_RETRY):
            outputs = generate_paraphrases(batch)
            if any(not is_exact_duplicate(o, a, seen) for o, a in zip(batch, outputs)):
                chosen_aug = outputs
                break

        if chosen_aug is None:
            idx += BATCH
            continue

        # Pair them and save
        for ori, aug in zip(batch, chosen_aug):

            if count >= target_100:
                break

            if not is_exact_duplicate(ori, aug, seen):

                pairs.append({
                    "original": ori,
                    "augmented": aug
                })

                seen.add(aug)
                count += 1
                pbar.update(1)

                if count % SAVE_INTERVAL == 0:
                    save_autosave(pairs)

                # milestone save
                for target, fname in milestones.items():
                    if count >= target and target not in saved_milestones:
                        need = target - len(originals)
                        aug_texts = [x["augmented"] for x in pairs[:need]]
                        df1_aug = pd.DataFrame({"label": 1, "Text": originals + aug_texts})
                        save_csv(df0, df1_aug, fname)
                        saved_milestones.add(target)

        idx += BATCH

    # Final save 100%
    need = target_100 - len(originals)
    aug_texts = [x["augmented"] for x in pairs[:need]]
    df1_aug = pd.DataFrame({"label": 1, "Text": originals + aug_texts})
    save_csv(df0, df1_aug, "t5_70_100.csv")
    save_autosave(pairs)
    pbar.close()

    print("\n===== Contoh 10 ORIGINAL vs AUGMENTED =====\n")
    for pair in pairs[:10]:
        print(f"[ORIGINAL ] {pair['original']}")
        print(f"[AUGMENTED] {pair['augmented']}\n")

# ================================
# RUN
# ================================
df = pd.read_csv(r"C:\SKRIPSI BRYAN\Tahap 2\2A\data_70_2A.csv")
df.columns = ["label", "Text"]
augment_label_1(df)


DEVICE: cuda


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 1014.43it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded autosave: 0 augmented pairs, total label 1 = 2310


T5 Paraphrase:  60%|██████    | 2776/4620 [01:28<05:20,  5.75aug/s]  

[SAVE] t5_70_60.csv created.


T5 Paraphrase:  80%|████████  | 3702/4620 [04:19<02:44,  5.59aug/s]

[SAVE] t5_70_80.csv created.


T5 Paraphrase: 100%|██████████| 4620/4620 [07:15<00:00,  5.31aug/s]

[SAVE] t5_70_100.csv created.
[SAVE] t5_70_100.csv created.

===== Contoh 10 ORIGINAL vs AUGMENTED =====

[ORIGINAL ] Aku minta maaf kalau aku terlihat putus asa, tapi aku bener-bener pengen menyerah.
[AUGMENTED] Maalum. Aku minta maaf kalau aku tampak sangat putus asu, tapi aku benar-benar bertekad untuk menyerah

[ORIGINAL ] ini antara sy bkl bunuh diri atau realistis
[AUGMENTED] Smg tidak ada perbedaan antara bunuh diri atau realistis.

[ORIGINAL ] capek bgt anjink mati aja kli y
[AUGMENTED] Itu capek bgt si Aninnk sekarat aja dia pergi

[ORIGINAL ] Mau jual ginjal, apa bunuh diri karena terjerat pinjol,, solusi pinjol apa #BLACKPINK_WORLDTOUR #51stBudgetSpeech #stockmarketcrash #LiverpoolRealMadrid
[AUGMENTED] Mereka akan melakukannya. #BLACKPINK_WORLDTOUR 51stBudgetSpeech - StockmarketCrashs, Liverpool Real Madrid vs Poinzay? Solusi apanya sekarang di sini: Apa itu investitor yang jahat mana ?

[ORIGINAL ] Sebenernya bundir itu bukan keinganan tapi gw pengenðŸ˜­ #depresi #ADHD #bu